# Karachi Crime — Preprocessing (Crime Type Normalization + Spatial/Temporal Features)

This notebook:
1) Loads `karachi_crime_dataset.csv` (same folder / root)  
2) **Normalizes CRIME_TYPE** by merging semantically duplicate labels (data-driven mapping)  
3) Creates clean spatial & temporal features for downstream **clustering + profiling**  
4) Saves a cleaned dataset as `karachi_crime_dataset_cleaned.csv`

## Outputs (saved in same folder)
- `karachi_crime_dataset_cleaned.csv`
- `crime_type_mapping_applied.csv` (audit table)


In [1]:
# 0) Imports
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path(".")
RANDOM_STATE = 42
pd.set_option("display.max_rows", 200)


In [2]:
# 1) Load dataset
df = pd.read_csv("Raw Data/karachi_crime_dataset.csv")
print("Loaded shape:", df.shape)
print("Columns:", list(df.columns))
df.head()


Loaded shape: (100000, 21)
Columns: ['INCIDENT_ID', 'TOWN', 'TOWN_RISK_LEVEL', 'TOWN_PRIORITY_RANK', 'SUBDIVISION', 'SUBDIVISION_RISK_LEVEL', 'SUBDIVISION_PRIORITY_RANK', 'SEVERITY_SCORE', 'SEVERITY', 'DATE', 'LATITUDE', 'LONGITUDE', 'CRIME_TYPE', 'IS_RED_ZONE', 'IS_ORANGE_ZONE', 'IS_YELLOW_ZONE', 'IS_GREEN_ZONE', 'IS_WHITE_ZONE', 'RISK_ZONE', 'SOURCE', 'RANK']


,INCIDENT_ID,TOWN,TOWN_RISK_LEVEL,TOWN_PRIORITY_RANK,SUBDIVISION,SUBDIVISION_RISK_LEVEL,SUBDIVISION_PRIORITY_RANK,SEVERITY_SCORE,SEVERITY,DATE,...,LONGITUDE,CRIME_TYPE,IS_RED_ZONE,IS_ORANGE_ZONE,IS_YELLOW_ZONE,IS_GREEN_ZONE,IS_WHITE_ZONE,RISK_ZONE,SOURCE,RANK
0,INCIDENT_000001,Lyari Town,High,1,Baghdadi,High,1,10,High,2024-12-15,...,66.9998,Gang Violence,1,0,0,0,0,Red,Synthetic,1
1,INCIDENT_000002,Lyari Town,High,1,Baghdadi,High,1,10,High,2020-12-03,...,66.9998,Murder,1,0,0,0,0,Red,Synthetic,1
2,INCIDENT_000003,Lyari Town,High,1,Baghdadi,High,1,10,High,2025-01-24,...,66.9998,Murder,1,0,0,0,0,Red,Synthetic,1
3,INCIDENT_000004,Lyari Town,High,1,Baghdadi,High,1,10,High,2023-02-18,...,66.9998,Gang Violence,1,0,0,0,0,Red,Synthetic,1
4,INCIDENT_000005,Lyari Town,High,1,Baghdadi,High,1,10,High,2025-04-13,...,66.9998,Gang Violence,1,0,0,0,0,Red,Synthetic,1


## 2) Inspect current crime types (before normalization)

In [3]:
# Crime type distribution (before)
if "CRIME_TYPE" not in df.columns:
    raise ValueError("Expected column CRIME_TYPE not found in dataset.")

crime_before = df["CRIME_TYPE"].astype(str).str.strip()
counts_before = crime_before.value_counts().sort_values(ascending=False)

print("Distinct crime labels (before):", counts_before.shape[0])
display(counts_before.to_frame("count").head(30))


Distinct crime labels (before): 27


,count
CRIME_TYPE,
Robbery,10269
Snatching,10051
Murder,8138
Theft,7634
Gang Violence,7247
Political Violence,5970
Assault,5907
Drug Possession,5706
Homicide,5668


## 3) Crime Type Normalization Mapping

Based on your dataset's actual labels, we apply conservative merges:

- **SNATCHING**: Snatching, Cellphone Snatching, Mobile Snatching, Street Snatching  
- **GUN_VIOLENCE**: Gun Violence, Shootings  
- **ROBBERY**: Robbery, Mugging  
- **THEFT**: Theft, Phone Theft  
- **MURDER**: Murder, Homicide, Targeted Killing  
- **GANG_VIOLENCE**: Gang Violence, Gang Activity, Drug Mafia Activity  
- **ASSAULT**: Assault, Violence  

Everything else stays as-is (unless you later decide to merge/drop ambiguous `Street Crimes`).


In [4]:
# 3) Define mapping (old -> new)
CRIME_TYPE_MAP = {
    # SNATCHING family
    "Snatching": "SNATCHING",
    "Cellphone Snatching": "SNATCHING",
    "Mobile Snatching": "SNATCHING",
    "Street Snatching": "SNATCHING",

    # Guns / shooting
    "Gun Violence": "GUN_VIOLENCE",
    "Shootings": "GUN_VIOLENCE",

    # Robbery family
    "Robbery": "ROBBERY",
    "Mugging": "ROBBERY",

    # Theft family
    "Theft": "THEFT",
    "Phone Theft": "THEFT",

    # Lethal crimes
    "Murder": "MURDER",
    "Homicide": "MURDER",
    "Targeted Killing": "MURDER",

    # Organized crime / gangs
    "Gang Violence": "GANG_VIOLENCE",
    "Gang Activity": "GANG_VIOLENCE",
    "Drug Mafia Activity": "GANG_VIOLENCE",

    # Generic violence
    "Assault": "ASSAULT",
    "Violence": "ASSAULT",
}

# Optional: normalize label formatting (trim + collapse spaces)
def normalize_label(x: str) -> str:
    x = str(x).strip()
    x = " ".join(x.split())
    return x

df["CRIME_TYPE_ORIG"] = df["CRIME_TYPE"].map(normalize_label)
df["CRIME_TYPE_NORM"] = df["CRIME_TYPE_ORIG"].map(CRIME_TYPE_MAP).fillna(df["CRIME_TYPE_ORIG"])

# Audit mapping usage
audit = (
    df[["CRIME_TYPE_ORIG", "CRIME_TYPE_NORM"]]
    .drop_duplicates()
    .sort_values(["CRIME_TYPE_NORM", "CRIME_TYPE_ORIG"])
    .reset_index(drop=True)
)

print("Distinct crime labels (after):", df["CRIME_TYPE_NORM"].nunique())
display(audit)


Distinct crime labels (after): 16


,CRIME_TYPE_ORIG,CRIME_TYPE_NORM
0,Assault,ASSAULT
1,Violence,ASSAULT
2,Drug Possession,Drug Possession
3,Ethnic Conflict,Ethnic Conflict
4,Extortion,Extortion
5,Drug Mafia Activity,GANG_VIOLENCE
6,Gang Activity,GANG_VIOLENCE
7,Gang Violence,GANG_VIOLENCE
8,Gun Violence,GUN_VIOLENCE
9,Shootings,GUN_VIOLENCE


## 4) Compare distributions (before vs after)

In [5]:
# 4) Compare counts before vs after
before = df["CRIME_TYPE_ORIG"].value_counts().rename("before_count")
after  = df["CRIME_TYPE_NORM"].value_counts().rename("after_count")

# For "before" side, show original counts; for "after", show merged counts
print("\nTop 30 (before):")
display(before.head(30).to_frame())

print("\nTop 30 (after):")
display(after.head(30).to_frame())

# Show how much each normalized class absorbed
absorbed = (
    df.groupby("CRIME_TYPE_NORM")["CRIME_TYPE_ORIG"]
      .nunique()
      .sort_values(ascending=False)
      .rename("num_original_labels_merged")
      .to_frame()
)

print("\nHow many original labels merged into each normalized class (top):")
display(absorbed.head(30))



Top 30 (before):


,before_count
CRIME_TYPE_ORIG,
Robbery,10269
Snatching,10051
Murder,8138
Theft,7634
Gang Violence,7247
Political Violence,5970
Assault,5907
Drug Possession,5706
Homicide,5668



Top 30 (after):


,after_count
CRIME_TYPE_NORM,
MURDER,14419
GANG_VIOLENCE,13274
SNATCHING,11330
ROBBERY,10910
THEFT,8022
ASSAULT,6256
Political Violence,5970
Drug Possession,5706
Land Grabbing,5269



How many original labels merged into each normalized class (top):


,num_original_labels_merged
CRIME_TYPE_NORM,
SNATCHING,4
GANG_VIOLENCE,3
MURDER,3
ASSAULT,2
THEFT,2
ROBBERY,2
GUN_VIOLENCE,2
Drug Possession,1
Industrial Theft,1


## 5) Spatial + Temporal feature cleanup (for clustering/profiling)

We create:
- `lat`, `lon` numeric
- `date_parsed` (best-effort)
- `hour`, `day_of_week`, `is_weekend`, `month` (if date/time exist)

If your dataset uses different date/time column names, this cell detects common variants.


In [ ]:
# ==============================
# 1) Drop leaky / post-hoc columns
# ==============================

LEAKY_COLS = [
    "RISK_ZONE",
    "IS_RED_ZONE", "IS_ORANGE_ZONE", "IS_YELLOW_ZONE",
    "IS_GREEN_ZONE", "IS_WHITE_ZONE",
    "SEVERITY_SCORE", "SEVERITY",
    "TOWN_RISK_LEVEL", "TOWN_PRIORITY_RANK",
    "SUBDIVISION_RISK_LEVEL", "SUBDIVISION_PRIORITY_RANK",
    "RANK"
]

existing_leaky = [c for c in LEAKY_COLS if c in df.columns]
print("Dropping leaky columns:", existing_leaky)

df = df.drop(columns=existing_leaky)
print("Remaining columns:", df.columns.tolist())


In [6]:
# 5) Spatial / Temporal features

# ---- Latitude/Longitude ----
lat_candidates = [c for c in df.columns if c.upper() in ["LATITUDE", "LAT"]]
lon_candidates = [c for c in df.columns if c.upper() in ["LONGITUDE", "LON", "LONG"]]

if not lat_candidates or not lon_candidates:
    print("⚠️ Could not auto-detect LAT/LON columns. Available columns:", list(df.columns))
else:
    LAT_COL = lat_candidates[0]
    LON_COL = lon_candidates[0]
    df["lat"] = pd.to_numeric(df[LAT_COL], errors="coerce")
    df["lon"] = pd.to_numeric(df[LON_COL], errors="coerce")
    print("Using lat/lon columns:", LAT_COL, LON_COL)
    print("Missing lat:", df["lat"].isna().mean().round(4), "Missing lon:", df["lon"].isna().mean().round(4))

# ---- Date/Time ----
date_candidates = [c for c in df.columns if c.upper() in ["DATE", "INCIDENT_DATE", "CRIME_DATE"]]
time_candidates = [c for c in df.columns if c.upper() in ["TIME", "INCIDENT_TIME", "CRIME_TIME"]]

DATE_COL = date_candidates[0] if date_candidates else None
TIME_COL = time_candidates[0] if time_candidates else None
print("Detected DATE col:", DATE_COL, "| TIME col:", TIME_COL)

# Parse datetime
dt = None
if DATE_COL and TIME_COL:
    dt = pd.to_datetime(df[DATE_COL].astype(str) + " " + df[TIME_COL].astype(str), errors="coerce")
elif DATE_COL:
    dt = pd.to_datetime(df[DATE_COL], errors="coerce")

df["datetime"] = dt

if df["datetime"].notna().any():
    df["hour"] = df["datetime"].dt.hour
    df["day_of_week"] = df["datetime"].dt.dayofweek  # Mon=0..Sun=6
    df["is_weekend"] = df["day_of_week"].isin([5,6]).astype(int)
    df["month"] = df["datetime"].dt.month
else:
    print("⚠️ datetime could not be parsed from available columns; temporal features not created.")

df[["CRIME_TYPE_ORIG","CRIME_TYPE_NORM","lat","lon","datetime","is_weekend","month"]].head(10)


Using lat/lon columns: LATITUDE LONGITUDE
Missing lat: 0.0 Missing lon: 0.0
Detected DATE col: DATE | TIME col: None


,CRIME_TYPE_ORIG,CRIME_TYPE_NORM,lat,lon,datetime,is_weekend,month
0,Gang Violence,GANG_VIOLENCE,24.8592,66.9998,2024-12-15,1,12
1,Murder,MURDER,24.8592,66.9998,2020-12-03,0,12
2,Murder,MURDER,24.8592,66.9998,2025-01-24,0,1
3,Gang Violence,GANG_VIOLENCE,24.8592,66.9998,2023-02-18,1,2
4,Gang Violence,GANG_VIOLENCE,24.8592,66.9998,2025-04-13,1,4
5,Gang Violence,GANG_VIOLENCE,24.8592,66.9998,2021-04-14,0,4
6,Kidnapping,Kidnapping,24.8592,66.9998,2021-02-28,1,2
7,Murder,MURDER,24.8592,66.9998,2020-10-06,0,10
8,Murder,MURDER,24.8592,66.9998,2022-12-03,1,12
9,Gang Violence,GANG_VIOLENCE,24.8592,66.9998,2022-06-19,1,6


## 6) Optional: Handle ambiguous `Street Crimes`

`Street Crimes` is semantically ambiguous (could include robbery/snatching/theft).
For an industry-ready system, you can either:
- keep it as its own class, or
- drop it from profiling, or
- re-map it after manual review.

By default: **keep it**.


In [10]:
# ==============================
# 1) Drop leaky / post-hoc columns
# ==============================

LEAKY_COLS = [
    "RISK_ZONE",
    "IS_RED_ZONE", "IS_ORANGE_ZONE", "IS_YELLOW_ZONE",
    "IS_GREEN_ZONE", "IS_WHITE_ZONE",
    "SEVERITY_SCORE", "SEVERITY",
    "TOWN_RISK_LEVEL", "TOWN_PRIORITY_RANK",
    "SUBDIVISION_RISK_LEVEL", "SUBDIVISION_PRIORITY_RANK",
    "RANK", 'INCIDENT_ID'
]

existing_leaky = [c for c in LEAKY_COLS if c in df.columns]
print("Dropping leaky columns:", existing_leaky)

df = df.drop(columns=existing_leaky)
print("Remaining columns:", df.columns.tolist())


Dropping leaky columns: ['INCIDENT_ID']
Remaining columns: ['TOWN', 'SUBDIVISION', 'DATE', 'LATITUDE', 'LONGITUDE', 'CRIME_TYPE', 'SOURCE', 'CRIME_TYPE_ORIG', 'CRIME_TYPE_NORM', 'lat', 'lon', 'datetime', 'hour', 'day_of_week', 'is_weekend', 'month']


In [7]:
# 6) Optional policy for Street Crimes
# Choose one: "keep", "drop"
STREET_CRIMES_POLICY = "keep"

if STREET_CRIMES_POLICY == "drop":
    before_rows = len(df)
    df = df[df["CRIME_TYPE_NORM"] != "Street Crimes"].copy()
    print(f"Dropped Street Crimes. Rows: {before_rows} -> {len(df)}")
else:
    print("Keeping Street Crimes as-is.")


Keeping Street Crimes as-is.


## 7) Save cleaned dataset + mapping audit

In [8]:
# 7) Save outputs
clean_cols = df.columns.tolist()

# Save mapping audit (unique pairs)
audit.to_csv(DATA_DIR / "crime_type_mapping_applied.csv", index=False)

# Save cleaned dataset
out_path = DATA_DIR / "karachi_crime_dataset_cleaned.csv"
df.to_csv(out_path, index=False)

print("✅ Saved:", out_path)
print("✅ Saved: crime_type_mapping_applied.csv")
print("Final rows:", len(df), "Final columns:", len(clean_cols))


✅ Saved: karachi_crime_dataset_cleaned.csv
✅ Saved: crime_type_mapping_applied.csv
Final rows: 100000 Final columns: 30
